In [58]:
!pip install bcrypt
import requests
import sqlite3
import bcrypt
import re

In [4]:
conn = sqlite3.connect("users.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS users (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    username TEXT UNIQUE NOT NULL,
    password_hash BLOB NOT NULL
)
""")
conn.commit()

In [5]:
def atbash(text):
    result = ""
    for char in text:
        if char.isalpha():
            if char.isupper():
                result += chr(90 - (ord(char) - 65))
            else:
                result += chr(122 - (ord(char) - 97))
        else:
            result += char
    return result

In [6]:
def caesar_encrypt(text, shift):
    result = ""
    for char in text:
        if char.isalpha():
            base = 65 if char.isupper() else 97
            result += chr((ord(char) - base + shift) % 26 + base)
        else:
            result += char
    return result

In [35]:
def caesar_decrypt(cipher, shift):
    return caesar_encrypt(cipher, -shift)

In [7]:
def vigenere_encrypt(text, key):
    result = ""
    key = key.upper()
    key_index = 0

    for char in text:
        if char.isalpha():
            base = 65 if char.isupper() else 97
            shift = ord(key[key_index % len(key)]) - 65
            result += chr((ord(char) - base + shift) % 26 + base)
            key_index += 1
        else:
            result += char
    return result

In [36]:
def vigenere_decrypt(cipher, key):
    result = ""
    key = key.upper()
    key_index = 0

    for char in cipher:
        if char.isalpha():
            base = 65 if char.isupper() else 97
            shift = ord(key[key_index % len(key)]) - 65
            result += chr((ord(char) - base - shift + 26) % 26 + base)
            key_index += 1
        else:
            result += char
    return result

In [71]:
def user_signup(username, password):
  cursor.execute("SELECT id FROM users WHERE username = ?", (username,))
  if cursor.fetchone():
      print("\n⚠️ Username already exists!\n")
      return

  if len(username) < 6:
    print("\n❌ Username must be at least 6 characters long!\n")
    return
  if len(password) < 8:
    print("\n❌ Password must be at least 8 characters long!\n")
    return
  if not re.search(r"[a-zA-Z]", password):
    print("\n❌ Password must contain at least one letter.\n")
    return
  if not re.search(r"\d", password):
    print("\n❌ Password must contain at least one number.\n")
    return
  if not re.search(r"[!@#$%^&*()_+=\-{};:'\",.<>/?`~]", password):
    print("\n❌ Password must contain at least one special character.\n")
    return

  hashed_pw = bcrypt.hashpw(password.encode("utf-8"), bcrypt.gensalt())

  cursor.execute("INSERT INTO users (username, password_hash) VALUES (?, ?)",
                   (username, hashed_pw))
  conn.commit()

  print("\n✅ User registered successfully!\n")

In [59]:
login_attempts = {}

def user_login(username, password):
    cursor.execute("SELECT password_hash FROM users WHERE username = ?", (username,))
    row = cursor.fetchone()

    if username not in login_attempts:
        login_attempts[username] = {"attempts": 0, "blocked": False}

    if login_attempts[username]["blocked"]:
        print("\n ⚠️ Account is locked due to too many failed attempts.\n")
        return False

    if not row:
        print("\n❌ Invalid username\n")
        return False

    if not bcrypt.checkpw(password.encode("utf-8"), row[0]):
        login_attempts[username]["attempts"] += 1
        remaining_attempts = 3 - login_attempts[username]["attempts"]

        if login_attempts[username]["attempts"] >= 3:
            login_attempts[username]["blocked"] = True
            print("\n⚠️ Incorrect password. Account is now locked.\n")
        else:
            print(f"\n❌ Incorrect password. {remaining_attempts} attempts remaining.\n")
        return False
    else:
      login_attempts[username] = {"attempts": 0, "blocked": False}
      print("\n✅ User Logged In successfully!\n")
      return True

In [72]:
while True:
    logged_in = False
    print("# ------------------ Welcome to D3CYPH3R! ------------------ #\n")
    print("1) Log In\n2) Sign Up\n0) Exit")
    choice1 = int(input("Enter a number: "))
    print(" ")

    if choice1 == 1:
      print("# --------------------- LOGIN --------------------- #\n")
      username = input("Enter Username: ")
      password = input("Enter Password: ")
      logged_in = user_login(username, password)
    elif choice1 == 2:
      print("# -------------------- REGISTER -------------------- #\n")
      username = input("Enter Username: ")
      password = input("Enter Password: ")
      user_signup(username, password)
      continue
    else:
      continue

    while logged_in == True:
      print("# ----------------- Welcome, " + username +"! ----------------- #\n")
      print("Choose a Cipher Method: \n1) Atbash\n2) Caesar Cipher\n3) Vigenere Cipher\n0) Logout")
      choice2 = int(input("Enter a number: "))

      if choice2 == 1:
        print("# -------------------- ATBASH -------------------- #\n")
        print("1) Encrypt\n2) Decrypt")
        atbash_choice = int(input("Choose mode: "))

        if atbash_choice == 1:
          text = input("\nEnter Text: ")
          result_text = atbash(text)
          print("\n✅ Done!\nResult Text: " + result_text + "\n")
          continue
        elif atbash_choice == 2:
          text = input("\nEnter Cipher: ")
          result_text = atbash(text)
          print("\n✅ Done!\nResult Text: " + result_text + "\n")
          continue
        else:
          print("\n❌ Invalid Selection!\n")
          continue
      elif choice2 == 2:
        print("# ----------------- CAESAR CIPHER ----------------- #\n")
        print("1) Encrypt\n2) Decrypt")
        caesar_choice = int(input("Choose mode: "))

        if caesar_choice == 1:
          text = input("\nEnter Text: ")
          key = int(input("Enter Key: "))
          result_text = caesar_encrypt(text, key)
          print("\n✅ Done!\nResult Text: " + result_text + "\n")
          continue
        elif caesar_choice == 2:
          text = input("\nEnter Cipher: ")
          key = int(input("Enter Key: "))
          result_text = caesar_decrypt(text, key)
          print("\n✅ Done!\nResult Text: " + result_text + "\n")
          continue
        else:
          print("\n❌ Invalid Selection!\n")
          continue
      elif choice2 == 3:
        print("# ---------------- VIGENERE CIPHER ---------------- #\n")
        print("1) Encrypt\n2) Decrypt")
        vigenere_choice = int(input("Choose mode: "))

        if vigenere_choice == 1:
          text = input("\nEnter Text: ")
          key = input("Enter Key: ")
          result_text = vigenere_encrypt(text, key)
          print("\n✅ Done!\nResult Text: " + result_text + "\n")
          continue
        elif vigenere_choice == 2:
          text = input("\nEnter Cipher: ")
          key = input("Enter Key: ")
          result_text = vigenere_decrypt(text, key)
          print("\n✅ Done!\nResult Text: " + result_text + "\n")
          continue
        else:
          print("\n❌ Invalid Selection!\n")
          continue
      elif choice2 == 0:
        print(" ")
        break
      else:
        continue




# ------------------ Welcome to D3CYPH3R! ------------------ #

1) Log In
2) Sign Up
0) Exit
Enter a number: 2
 
# -------------------- REGISTER -------------------- #

Enter Username: babybabybaby
Enter Password: babybabybaby123

❌ Password must contain at least one special character.

# ------------------ Welcome to D3CYPH3R! ------------------ #

1) Log In
2) Sign Up
0) Exit
Enter a number: 2
 
# -------------------- REGISTER -------------------- #

Enter Username: bbybbybby
Enter Password: bbybbybby

❌ Password must contain at least one number.

# ------------------ Welcome to D3CYPH3R! ------------------ #

1) Log In
2) Sign Up
0) Exit


KeyboardInterrupt: Interrupted by user